In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoProcessor, AutoModelForCausalLM 
import os
from pathlib import Path
from PIL import Image


In [ ]:
# Set device and torch dtype
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Using device: {device}")
print(f"Using dtype: {torch_dtype}")


In [ ]:
# Initialize Florence-2 model
florence_model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-large", torch_dtype=torch_dtype, trust_remote_code=True).to(device)
processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

print("Florence-2 model loaded successfully")


In [ ]:
def run_florence_inference(image, task_prompt, text_input=None):
    """
    Run Florence-2 inference on an image with given task prompt
    """
    if text_input is None:
        prompt = task_prompt
    else:
        prompt = task_prompt + text_input
        
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device, torch_dtype)
    
    generated_ids = florence_model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3,
        do_sample=False
    )
    
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed_answer = processor.post_process_generation(generated_text, task=task_prompt, image_size=(image.width, image.height))
    
    return parsed_answer


In [ ]:
# Test data path
test_image_dir = "/Users/theo.moreau/Documents/futur/datasets/mvtec_anomaly_detection/screw/test"

# Load test image
image_path = os.path.join(test_image_dir, "good", os.listdir(os.path.join(test_image_dir, "good"))[0])
image_pil = Image.open(image_path).convert('RGB')

print(f"Loaded image: {image_path}")
print(f"Image size: {image_pil.size}")


In [ ]:
# Task 1: Caption to Phrase Grounding
task_prompt = "<CAPTION_TO_PHRASE_GROUNDING>"
text_input = "a screw"

result = run_florence_inference(image_pil, task_prompt, text_input)
print("Caption to Phrase Grounding Result:")
print(result)

# Visualize results
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(image_pil)
ax.set_title(f"Florence-2: {task_prompt} - '{text_input}'")

# Draw bounding boxes if any
if '<CAPTION_TO_PHRASE_GROUNDING>' in result:
    bboxes = result['<CAPTION_TO_PHRASE_GROUNDING>']['bboxes']
    labels = result['<CAPTION_TO_PHRASE_GROUNDING>']['labels']
    
    for bbox, label in zip(bboxes, labels):
        x1, y1, x2, y2 = bbox
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='red', linewidth=2)
        ax.add_patch(rect)
        ax.text(x1, y1-10, label, color='red', fontsize=12, weight='bold')

ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Object Detection
task_prompt = "<OD>"

result = run_florence_inference(image_pil, task_prompt)
print("Object Detection Result:")
print(result)

# Visualize object detection results
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(image_pil)
ax.set_title("Florence-2: Object Detection")

# Draw bounding boxes if any
if '<OD>' in result:
    bboxes = result['<OD>']['bboxes']
    labels = result['<OD>']['labels']
    
    for bbox, label in zip(bboxes, labels):
        x1, y1, x2, y2 = bbox
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='blue', linewidth=2)
        ax.add_patch(rect)
        ax.text(x1, y1-10, label, color='blue', fontsize=12, weight='bold')

ax.axis('off')
plt.tight_layout()
plt.show()
